In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import kagglehub
import os
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
df_delivary_data = pd.read_csv(path + '/Q1_data.csv')


In [ ]:
# Task 2: Write your code here:
df_delivary_data.head()

In [ ]:
# Task 3: Write your code here:
df_delivary_data.info()

In [ ]:
# Task 4: Write your code here:
df_delivary_data.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df_delivary_data, "Delivery_Time")

In [ ]:
df_delivary_data.drop(columns='Order_ID')

In [ ]:
Missing_data = (df_delivary_data.isnull().sum() / len(df_delivary_data))
df_Missing_data = pd.DataFrame(
    {
        'column':Missing_data.index,
        'Percent':Missing_data.values
    }
)
df_Missing_data = df_Missing_data[df_Missing_data['Percent'] > 0].sort_values('Percent',ascending=False)

Missing_cols = ['Delivery_Time','Weather','Traffic_Level','Time_of_Day','Courier_Experience_yrs']
print(f"Shape before cleaning: {df_delivary_data.shape}")
df_clean = df_delivary_data.dropna(subset=Missing_cols)
print(f"Shape after cleaning: {df_clean.shape}")

In [ ]:
df_clean.drop_duplicates(inplace=True)
df_clean.shape

In [ ]:

cat_cols = ['Weather','Traffic_Level','Time_of_Day','Vehicle_Type']
le = LabelEncoder()
for cols in cat_cols:
  df_clean[cols] = le.fit_transform(df_clean[cols])
df_clean.head()

In [ ]:
scaler = StandardScaler()
numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
X = df_clean.drop("Delivery_Time", axis=1)
y = df_clean['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse_scores = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{5}")
  print(f"MAE: {mse:,.2f}")
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics
  mse = mean_squared_error(y_test, y_pred)

  # Store results
  mse_scores.append(mse)
mse_scores = np.array(mse_scores)
print(f"5-Fold MSE Results:")
print(f"MAE: {mse_scores.mean():,.2f}")



In [ ]:
feature_importance = pd.DataFrame({
    'feature':numerical_cols ,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: